# Q3 - Semantic Candidate Generation (Embeddings)

Loads the article embeddings computed on Kaggle (see
`src/compute_embeddings_kaggle.ipynb`), retrieves top-K candidates per user
via mean-pooled click history and brute-force cosine similarity, reports
recall@K, and compares against Q2's BM25 results per SPEC.md's Q3 section.

Run top-to-bottom (or via `python embedding_retrieval.py`) to rebuild
`data/processed/{dataset}/embedding_topk.parquet` and `embedding_metrics.json`.

Loading/filtering uses `polars` throughout (same reasoning as
`src/build_pipeline.ipynb`/`src/bm25_retrieval.ipynb`), with the same
`BUILD_LARGE_ONLY` flag convention and progress appended to
`build_progress.log`. Requires `data/processed/{dataset}/article_embeddings.parquet`
to already exist for every dataset in scope -- for `ebnerd_large`/`mind_large`
that means running `src/compute_embeddings_kaggle.ipynb` on Kaggle first.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import shutil

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.embeddings import mean_pool, batched_top_k


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"

# Same flag/convention as src/build_pipeline.ipynb and src/bm25_retrieval.ipynb.
BUILD_LARGE_ONLY = True
DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


RECENT_N_CLICKS = 20
CANDIDATE_K_VALUES = [50, 100, 200]
TOPK_MAX = max(CANDIDATE_K_VALUES)
# Smaller than the original 2000 -- np.argpartition's own scratch array is
# (BATCH_SIZE, n_docs) int64, and ebnerd_large's 125,541-doc corpus makes
# that allocation ~2x bigger than what 2000 was sized for on the smaller
# datasets (max ~65,238 docs) this constant was originally tuned against.
BATCH_SIZE = 500
EMBEDDING_MODEL_NAME = "paraphrase-xlm-r-multilingual-v1"  # computed on Kaggle, see src/compute_embeddings_kaggle.ipynb

log_progress(f"embedding_retrieval started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY}, datasets={DATASETS})")

feature_store = {}
for name in DATASETS:
    feature_store[name] = {
        "articles": pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id"]),
        "behaviors": pl.read_parquet(DATA_DIR / name / "behaviors.parquet", columns=["user_id", "article_ids_clicked", "split"]),
        "history": pl.read_parquet(DATA_DIR / name / "history.parquet", columns=["user_id", "article_id_sequence"]),
    }

embedding_paths = {name: DATA_DIR / name / "article_embeddings.parquet" for name in DATASETS}
missing = [str(p) for p in embedding_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"missing article_embeddings.parquet: {missing}. "
        "Run src/compute_embeddings_kaggle.ipynb on Kaggle first and place the "
        "downloaded files at these paths (see README.md)."
    )

embeddings_raw = {name: pl.read_parquet(embedding_paths[name]) for name in DATASETS}
for name in DATASETS:
    log_progress(
        f"  {name}: loaded articles/behaviors/history/embeddings "
        f"({feature_store[name]['articles'].height}, {feature_store[name]['behaviors'].height}, "
        f"{feature_store[name]['history'].height}, {embeddings_raw[name].height} rows)"
    )

{name: {table: df.shape for table, df in tables.items()} for name, tables in feature_store.items()}

{'ebnerd_large': {'articles': (125541, 1),
  'behaviors': (24630275, 3),
  'history': (974791, 2)},
 'mind_large': {'articles': (104151, 1),
  'behaviors': (2609219, 3),
  'history': (750434, 2)}}

## Build corpus embedding matrices

Reindexed to match `articles.parquet`'s `article_id` order explicitly
(not assumed from `article_embeddings.parquet`'s row order), so downstream
`doc_ids`/`matrix` alignment is correct regardless of how the Kaggle notebook
wrote its output.

In [2]:
corpus = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    emb_ids = embeddings_raw[name]["article_id"].to_list()
    emb_vectors = embeddings_raw[name]["embedding"].to_list()
    emb_lookup = {aid: np.asarray(vec) for aid, vec in zip(emb_ids, emb_vectors)}
    doc_ids = articles["article_id"].to_numpy()

    missing_ids = set(doc_ids) - set(emb_lookup)
    if missing_ids:
        raise ValueError(f"{name}: {len(missing_ids)} articles have no embedding")

    matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
    corpus[name] = {"doc_ids": doc_ids, "matrix": matrix, "embedding_lookup": emb_lookup}
    log_progress(f"  {name}: embedding matrix built {matrix.shape}")

{name: c["matrix"].shape for name, c in corpus.items()}

{'ebnerd_large': (125541, 768), 'mind_large': (104151, 768)}

In [3]:
def test_corpus_embeddings_aligned():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        c = corpus[name]
        assert c["matrix"].shape[0] == len(articles)
        assert (c["doc_ids"] == articles["article_id"].to_numpy()).all()
        assert not np.isnan(c["matrix"]).any()
        assert set(embeddings_raw[name]["article_id"].to_list()) == set(articles["article_id"].to_list())


test_corpus_embeddings_aligned()
print("ok: corpus embedding matrices aligned with articles.parquet, no NaNs")

ok: corpus embedding matrices aligned with articles.parquet, no NaNs


## Mean-pooling user representation

Same recency window as Q2's BM25 query construction (`sequence[-20:]`, the
*last* N clicks) -- kept identical so the lexical-vs-semantic comparison
holds the input click window constant.

In [4]:
def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)

In [5]:
def test_build_user_query_vector():
    assert build_user_query_vector([], {}) is None

    lookup = {"a": np.array([1.0, 0.0]), "b": np.array([0.0, 1.0]), "c": np.array([2.0, 0.0])}
    # recent_n=1 should only see the *last* id ("c"), not "a"
    vec = build_user_query_vector(["a", "b", "c"], lookup, recent_n=1)
    assert np.allclose(vec, [2.0, 0.0])

    vec_all = build_user_query_vector(["a", "b", "c"], lookup, recent_n=10)
    assert np.allclose(vec_all, np.mean([lookup["a"], lookup["b"], lookup["c"]], axis=0))


test_build_user_query_vector()
print("ok: mean-pooling takes the most recent N clicks (last N, not first N)")

ok: mean-pooling takes the most recent N clicks (last N, not first N)


## Top-K retrieval smoke test

`batched_top_k` is exact within one call (see its docstring for a documented
float32 caveat about comparing *separate* calls with different batch
shapes) -- this test only ever compares results from a single, consistently-shaped
call, which is also how the real per-user retrieval cache below uses it.

In [6]:
def test_batched_top_k():
    name = "mind" if "mind" in DATASETS else DATASETS[0]
    c = corpus[name]

    # self-similarity: a 'user' whose only click is doc 0 should retrieve doc 0 first, score ~= 1.0
    query = c["matrix"][0:1]
    top5 = batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=BATCH_SIZE)[0]
    assert top5[0][0] == c["doc_ids"][0]
    assert abs(top5[0][1] - 1.0) < 1e-3

    scores = [s for _, s in top5]
    assert scores == sorted(scores, reverse=True)

    # nesting invariant -- both calls use the same single-query batch shape (see docstring caveat)
    top200 = batched_top_k(query, c["matrix"], c["doc_ids"], k=200, batch_size=BATCH_SIZE)[0]
    top50 = batched_top_k(query, c["matrix"], c["doc_ids"], k=50, batch_size=BATCH_SIZE)[0]
    assert top200[:50] == top50

    assert batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=BATCH_SIZE) == \
           batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=1)


test_batched_top_k()
print("ok: top-K retrieval is sorted, self-similarity sane, and nested within a call")

ok: top-K retrieval is sorted, self-similarity sane, and nested within a call


## Per-user retrieval cache (val/test users only)

Same cold-start definition Q2 uses (empty `article_id_sequence`) and the
same val/test-only scope -- recomputed here (separate notebook process) but
cross-checked below against Q2's already-persisted `bm25_metrics.json`
counts, which must match exactly since both come from the same `behaviors`/
`history` tables.

In [7]:
QUERY_CHUNK_SIZE = 50_000  # bounds batched_top_k's own results-list accumulation per call (see markdown below)

user_topk = {}
coldstart_users = {}
for name in DATASETS:
    behaviors = feature_store[name]["behaviors"]
    history = feature_store[name]["history"]
    eval_user_ids = set(behaviors.filter(pl.col("split").is_in(["val", "test"]))["user_id"].to_list())
    history_eval = history.filter(pl.col("user_id").is_in(list(eval_user_ids)))
    lookup = corpus[name]["embedding_lookup"]

    query_user_ids, query_vectors = [], []
    coldstart = set()
    user_ids_col = history_eval["user_id"].to_list()
    sequences_col = history_eval["article_id_sequence"].to_list()
    n_users = len(user_ids_col)
    log_progress(f"  {name}: building mean-pooled query vectors for {n_users} eval users")
    for i, (user_id, article_id_sequence) in enumerate(zip(user_ids_col, sequences_col)):
        vec = build_user_query_vector(article_id_sequence, lookup)
        if vec is None:
            coldstart.add(user_id)
            continue
        query_user_ids.append(user_id)
        query_vectors.append(vec)
        if (i + 1) % 50_000 == 0:
            log_progress(f"    {name}: {i + 1}/{n_users} users processed")

    query_matrix = np.stack(query_vectors).astype(np.float32) if query_vectors else np.zeros((0, corpus[name]["matrix"].shape[1]))
    n_queries = len(query_user_ids)
    log_progress(f"  {name}: batched cosine-similarity top-K over {n_queries} queries x {corpus[name]['matrix'].shape[0]} docs")

    # batched_top_k processes queries in small batches internally (batch_size),
    # but accumulates ALL results into one Python list before returning -- at
    # ebnerd_large scale (821,111 queries x 200-item results) that list alone
    # grows to ~164M tuples, leaving no headroom for a later batch's own
    # np.argpartition scratch array (a comparatively modest ~1.9GB) and OOM'ing
    # partway through. Calling it in QUERY_CHUNK_SIZE-sized outer chunks instead
    # bounds that accumulation to one chunk at a time, freed between chunks.
    # No gc.collect() here -- del alone frees chunk_results via refcounting
    # (it's a plain, non-cyclic list of tuples); a forced gc.collect() would
    # scan this notebook's entire large live object graph on every chunk,
    # which cost real time for no memory benefit (see write_topk_parquet_chunked).
    topk_for_dataset = {}
    for start in range(0, n_queries, QUERY_CHUNK_SIZE):
        end = min(start + QUERY_CHUNK_SIZE, n_queries)
        chunk_results = batched_top_k(
            query_matrix[start:end], corpus[name]["matrix"], corpus[name]["doc_ids"], k=TOPK_MAX, batch_size=BATCH_SIZE
        )
        for user_id, result in zip(query_user_ids[start:end], chunk_results):
            topk_for_dataset[user_id] = result
        del chunk_results
        log_progress(f"    {name}: top-K computed for {end}/{n_queries} queries")

    user_topk[name] = topk_for_dataset
    coldstart_users[name] = coldstart
    log_progress(f"  {name}: retrieval done -- {len(user_topk[name])} retrieved, {len(coldstart)} cold-start")

{name: {"retrieved": len(user_topk[name]), "coldstart": len(coldstart_users[name])} for name in DATASETS}

{'ebnerd_large': {'retrieved': 821111, 'coldstart': 0},
 'mind_large': {'retrieved': 415122, 'coldstart': 10423}}

In [8]:
def test_user_retrieval_cache():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        eval_user_ids = set(behaviors.filter(pl.col("split").is_in(["val", "test"]))["user_id"].to_list())
        assert set(user_topk[name]).issubset(eval_user_ids)
        assert set(coldstart_users[name]).issubset(eval_user_ids)
        assert set(user_topk[name]) | set(coldstart_users[name]) == eval_user_ids

        for aid_scores in list(user_topk[name].values())[:5]:
            assert 1 <= len(aid_scores) <= TOPK_MAX

        # cross-check against Q2's already-persisted cold-start counts -- same
        # users, same criterion, computed independently in a separate notebook
        bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
        if bm25_metrics_path.exists():
            bm25_metrics = json.loads(bm25_metrics_path.read_text())
            for split in ["val", "test"]:
                expected_coldstart_impressions = bm25_metrics["n_impressions"][split]["excluded_coldstart"]
                actual_coldstart_impressions = behaviors.filter(
                    (pl.col("split") == split) & (pl.col("user_id").is_in(list(coldstart_users[name])))
                ).height
                assert actual_coldstart_impressions == expected_coldstart_impressions, (
                    f"{name}/{split}: cold-start impression count diverged from Q2's bm25_metrics.json"
                )


test_user_retrieval_cache()
print("ok: retrieval cache covers exactly the val/test user population, cold-start matches Q2's numbers")

ok: retrieval cache covers exactly the val/test user population, cold-start matches Q2's numbers


## Recall@K evaluation

Same fractional multi-relevant definition as Q2 (`article_ids_clicked` is
not always singleton) -- retrieval-method-independent, reused directly.

In [9]:
def evaluate_recall(dataset: str, split: str, k_values: list[int]) -> dict:
    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors.filter(pl.col("split") == split)
    topk = user_topk[dataset]
    coldstart = coldstart_users[dataset]

    recalls = {k: [] for k in k_values}
    n_excluded = 0

    for user_id, clicked in zip(split_behaviors["user_id"].to_list(), split_behaviors["article_ids_clicked"].to_list()):
        if user_id in coldstart:
            n_excluded += 1
            continue
        clicked = set(clicked)
        candidate_ids = [aid for aid, _ in topk[user_id]]
        for k in k_values:
            top_k_ids = set(candidate_ids[:k])
            recalls[k].append(len(clicked & top_k_ids) / len(clicked))

    n_evaluated = len(split_behaviors) - n_excluded
    return {
        "recall_at_k": {k: (sum(v) / len(v) if v else 0.0) for k, v in recalls.items()},
        "n_total": len(split_behaviors),
        "n_evaluated": n_evaluated,
        "n_excluded_coldstart": n_excluded,
    }


metrics = {name: {split: evaluate_recall(name, split, CANDIDATE_K_VALUES) for split in ["val", "test"]} for name in DATASETS}
log_progress(f"embedding_retrieval: recall@K computed for {DATASETS}")
metrics

{'ebnerd_large': {'val': {'recall_at_k': {50: 0.000847355164328057,
    100: 0.0016041101718554042,
    200: 0.0028535922510510786},
   'n_total': 1678989,
   'n_evaluated': 1678989,
   'n_excluded_coldstart': 0},
  'test': {'recall_at_k': {50: 0.0006466739613861877,
    100: 0.0011383464064127595,
    200: 0.002080821774892504},
   'n_total': 12566385,
   'n_evaluated': 12566385,
   'n_excluded_coldstart': 0}},
 'mind_large': {'val': {'recall_at_k': {50: 0.00838211980030798,
    100: 0.012926764720605022,
    200: 0.01991119985176957},
   'n_total': 431517,
   'n_evaluated': 420124,
   'n_excluded_coldstart': 11393},
  'test': {'recall_at_k': {50: 0.005300929828484517,
    100: 0.008881657752964239,
    200: 0.015166714622445912},
   'n_total': 376471,
   'n_evaluated': 365201,
   'n_excluded_coldstart': 11270}}}

In [10]:
def test_recall_at_k_monotonic():
    for name in DATASETS:
        for split in ["val", "test"]:
            m = metrics[name][split]
            r = m["recall_at_k"]
            assert r[50] <= r[100] + 1e-12 <= r[200] + 1e-12
            assert all(0.0 <= v <= 1.0 for v in r.values())
            assert m["n_evaluated"] + m["n_excluded_coldstart"] == m["n_total"]


test_recall_at_k_monotonic()
print("ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent")

ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent


## Lexical vs. semantic comparison

Reads both `bm25_metrics.json` (Q2) and this notebook's `metrics` (embeddings)
to compare recall@K side by side, per dataset/split.

In [11]:
comparison_rows = []
for name in DATASETS:
    bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
    bm25_metrics = json.loads(bm25_metrics_path.read_text()) if bm25_metrics_path.exists() else None
    for split in ["val", "test"]:
        for k in CANDIDATE_K_VALUES:
            row = {
                "dataset": name,
                "split": split,
                "k": k,
                "embedding_recall": metrics[name][split]["recall_at_k"][k],
                "bm25_recall": bm25_metrics["recall_at_k"][split][str(k)] if bm25_metrics else None,
            }
            comparison_rows.append(row)

comparison_table = pl.DataFrame(comparison_rows)
comparison_table

dataset,split,k,embedding_recall,bm25_recall
str,str,i64,f64,f64
"""ebnerd_large""","""val""",50,0.000847,0.001288
"""ebnerd_large""","""val""",100,0.001604,0.002598
"""ebnerd_large""","""val""",200,0.002854,0.005289
"""ebnerd_large""","""test""",50,0.000647,0.002164
"""ebnerd_large""","""test""",100,0.001138,0.003725
…,…,…,…,…
"""mind_large""","""val""",100,0.012927,0.019237
"""mind_large""","""val""",200,0.019911,0.029668
"""mind_large""","""test""",50,0.005301,0.003767


In [12]:
def test_comparison_table():
    assert len(comparison_table) == len(DATASETS) * 2 * len(CANDIDATE_K_VALUES)
    assert set(comparison_table["dataset"].to_list()) == set(DATASETS)
    if comparison_table["bm25_recall"].null_count() == 0:
        for name in DATASETS:
            bm25_metrics = json.loads((DATA_DIR / name / "bm25_metrics.json").read_text())
            for split in ["val", "test"]:
                for k in CANDIDATE_K_VALUES:
                    row = comparison_table.filter(
                        (pl.col("dataset") == name) & (pl.col("split") == split) & (pl.col("k") == k)
                    ).row(0, named=True)
                    assert abs(row["bm25_recall"] - bm25_metrics["recall_at_k"][split][str(k)]) < 1e-12


test_comparison_table()
print("ok: lexical-vs-semantic comparison table matches both persisted metrics files")

ok: lexical-vs-semantic comparison table matches both persisted metrics files


## Persist embedding outputs

Schema-parallel to Q2's `bm25_topk.parquet`/`bm25_metrics.json` (see
`SPEC.md` Q3 section 6).

In [13]:
import pyarrow.parquet as pq


def write_topk_parquet_chunked(user_ids: list, topk_lists: list, out_path: Path, dataset: str, chunk_rows: int = 50_000) -> None:
    """Same chunked-write pattern as build_pipeline.ipynb's write_parquet_chunked
    / bm25_retrieval.ipynb's write_topk_parquet_chunked (see SPEC.md Q1 #5,
    Q2 #10) -- building the whole pl.DataFrame in one shot OOM'd
    bm25_retrieval.ipynb at ebnerd_large scale (821,111 rows x two
    200-element list columns). Each chunk is written as its own small
    parquet file. The merge step uses pyarrow's ParquetWriter directly
    (append one row group per chunk file) rather than
    polars.scan_parquet(...).sink_parquet(...): the latter *also* OOM'd in
    bm25_retrieval.ipynb merging just 17 chunk files, because this polars
    version's sink_parquet still collects the full result into memory
    before writing rather than truly streaming -- pyarrow's row-group-append
    never holds more than one chunk's worth of data in memory regardless of
    the final file's total size.

    No explicit gc.collect() here (unlike an earlier version of this
    function): each chunk_df/table is a plain, non-cyclic object, so `del`
    alone frees it immediately via refcounting. A forced gc.collect() scans
    the *entire* live object graph for cycles -- with this notebook's large
    persistent state (corpus matrices, embedding lookups, the full
    user_topk dict), that full-heap scan turned out to cost minutes per
    call, not the near-zero cost it has on a small heap: it's what made the
    write step take ~2+ hours for ebnerd_large instead of the few minutes
    the actual I/O requires."""
    n = len(user_ids)
    if n <= chunk_rows:
        pl.DataFrame({
            "user_id": user_ids,
            "dataset": [dataset] * n,
            "n_retrieved": [len(x) for x in topk_lists],
            "retrieved_article_ids": [[aid for aid, _ in x] for x in topk_lists],
            "retrieved_scores": [[float(s) for _, s in x] for x in topk_lists],
        }).write_parquet(out_path)
        return

    tmp_dir = out_path.parent / f"_{out_path.stem}_chunks_tmp"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    n_chunks = (n + chunk_rows - 1) // chunk_rows
    try:
        for i, start in enumerate(range(0, n, chunk_rows)):
            end = min(start + chunk_rows, n)
            chunk_topk = topk_lists[start:end]
            chunk_df = pl.DataFrame({
                "user_id": user_ids[start:end],
                "dataset": [dataset] * (end - start),
                "n_retrieved": [len(x) for x in chunk_topk],
                "retrieved_article_ids": [[aid for aid, _ in x] for x in chunk_topk],
                "retrieved_scores": [[float(s) for _, s in x] for x in chunk_topk],
            })
            chunk_df.write_parquet(tmp_dir / f"part_{i:04d}.parquet")
            del chunk_df
            log_progress(f"    {out_path.name}: chunk {i + 1}/{n_chunks} written")

        writer = None
        try:
            for chunk_path in sorted(tmp_dir.glob("part_*.parquet")):
                table = pq.read_table(chunk_path)
                if writer is None:
                    writer = pq.ParquetWriter(out_path, table.schema)
                writer.write_table(table)
                del table
        finally:
            if writer is not None:
                writer.close()
        log_progress(f"    {out_path.name}: merged {n_chunks} chunks")
    finally:
        shutil.rmtree(tmp_dir)


def write_embedding_outputs(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    user_ids = list(user_topk[dataset].keys())
    topk_lists = list(user_topk[dataset].values())
    write_topk_parquet_chunked(user_ids, topk_lists, out_dir / "embedding_topk.parquet", dataset)
    log_progress(f"  {dataset}: wrote embedding_topk.parquet ({len(user_ids)} rows)")

    embedding_metrics = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "embedding_method": EMBEDDING_MODEL_NAME,
            "embedding_dim": int(corpus[dataset]["matrix"].shape[1]),
            "recent_n_clicks": RECENT_N_CLICKS,
            "topk_max": TOPK_MAX,
        },
        "recall_at_k": {split: metrics[dataset][split]["recall_at_k"] for split in ["val", "test"]},
        "n_impressions": {
            split: {
                "total": metrics[dataset][split]["n_total"],
                "evaluated": metrics[dataset][split]["n_evaluated"],
                "excluded_coldstart": metrics[dataset][split]["n_excluded_coldstart"],
            }
            for split in ["val", "test"]
        },
        "scope": "val_test_users_only",
    }
    (out_dir / "embedding_metrics.json").write_text(json.dumps(embedding_metrics, indent=2))
    log_progress(f"  {dataset}: wrote embedding_metrics.json")
    return out_dir


embedding_out_dirs = {name: write_embedding_outputs(name) for name in DATASETS}
log_progress("embedding_retrieval: all outputs written")
embedding_out_dirs

{'ebnerd_large': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd_large'),
 'mind_large': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/mind_large')}

In [14]:
def test_embedding_outputs_roundtrip():
    for name in DATASETS:
        out_dir = embedding_out_dirs[name]
        topk_path = out_dir / "embedding_topk.parquet"
        metrics_path = out_dir / "embedding_metrics.json"
        assert topk_path.exists() and metrics_path.exists()

        reloaded_topk = pl.read_parquet(topk_path)
        assert set(reloaded_topk["user_id"].to_list()) == set(user_topk[name])
        for row in reloaded_topk.head(20).iter_rows(named=True):
            assert len(row["retrieved_article_ids"]) == row["n_retrieved"]
            assert len(row["retrieved_scores"]) == row["n_retrieved"]

        reloaded_metrics = json.loads(metrics_path.read_text())
        for split in ["val", "test"]:
            for k in CANDIDATE_K_VALUES:
                expected = metrics[name][split]["recall_at_k"][k]
                actual = reloaded_metrics["recall_at_k"][split][str(k)]
                assert abs(expected - actual) < 1e-12


test_embedding_outputs_roundtrip()
log_progress("embedding_retrieval completed successfully")
print("ok: embedding_topk.parquet and embedding_metrics.json round-trip correctly for both datasets")

ok: embedding_topk.parquet and embedding_metrics.json round-trip correctly for both datasets
